### **Rôle principal du moteur CDS :**
- Ce fichier agit comme le cœur du Système d'Aide à la Décision Clinique (Clinical Decision Support).
- Il a pour mission d'analyser les lignes d'une prescription médicale et de les confronter au profil clinique du patient (âge, allergies, grossesse, fonction rénale) afin de générer des alertes de sécurité (CdsAlert).


### **Étapes et règles cliniques implémentées :**

##### **Étape 1 : Préparation et requêtes groupées (Batch Loading)**

- Le moteur commence par calculer l'âge du patient et normaliser le texte des molécules (gestion de la casse et des accents).
- Pour optimiser les performances, il charge en une seule fois depuis la base de données toutes les informations relatives aux médicaments prescrits, y compris leurs DCI (Dénominations Communes Internationales) primaires et secondaires.

##### **Étape 2 : Règle 1 — Détection des allergies (ALLERGY)**

Le système évalue le risque allergique sur trois niveaux d'alerte :

- Correspondance directe : Vérifie si la molécule prescrite correspond exactement à une allergie connue du patient.
- Recherche textuelle : Scrutent le texte libre des fiches médicaments (contre-indications, excipients) pour y trouver le nom de l'allergène.
- Allergies croisées : Utilise un dictionnaire interne (ALLERGY_FAMILY_MAP) pour lier des familles de médicaments ; par exemple, signaler un risque avec l'"amoxicilline" pour un patient allergique à la "pénicilline".

##### **Étape 3 : Règle 2 — Redondance de molécules (REDUNDANT_DCI)**

- Le script isole la DCI principale de chaque médicament et vérifie si cette même molécule est prescrite plusieurs fois sous des noms commerciaux différents, générant une alerte pour risque de surdosage cumulatif.

##### **Étape 4 : Règles 3 & 4 — Âge et profils spécifiques (POSOLOGY & CONTRA_INDICATION)**

- Il vérifie que l'âge du patient est supérieur à l'âge minimum d'utilisation défini pour le médicament.
- Si le dossier indique que la patiente est enceinte ou allaitante, le moteur cherche des mots-clés spécifiques (ex: "grossesse", "tératogène", "lait maternel") dans la liste des contre-indications de la molécule.

##### **Étape 5 : Règle 5 — Information psychoactive (PSYCHOACTIVE)**

- Il génère une alerte mineure purement informative pour alerter le prescripteur si le médicament contient des substances psychoactives nécessitant une vigilance sur les co-prescriptions sédatives.

##### **Étape 6 : Règle 6 — Interactions médicamenteuses (INTERACTION)**

- La fonction dédiée _check_interactions croise l'ensemble des DCI présentes sur l'ordonnance et interroge la base de données (issues du Thésaurus ANSM) en une seule requête SQL pour identifier toute paire médicamenteuse dangereuse.

##### **Étape 7 : Règle 7 — Insuffisance rénale (RENAL)**

- Si la clairance de la créatinine du patient est connue, le script la compare à un dictionnaire codé en dur (RENAL_RISK_DRUGS) contenant des seuils critiques.
- Il adapte la sévérité de l'alerte (nécessité de baisser la dose ou contre-indication absolue) selon le stade de l'insuffisance rénale et la toxicité du médicament (ex: Metformine, AINS, certains antibiotiques).